In [ ]:
# Análisis de Resultados y Validación
## Impacto Operativo y Económicoimport pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Cargar datos
df = pd.read_excel('../data/raw/dataset_sintetico_demanda_lima.xlsx')
df['fecha'] = pd.to_datetime(df['fecha'])

In [ ]:
# Parámetros económicos
UNIT_COST = 10.0      # Costo por unidad de producción
SELLING_PRICE = 25.0  # Precio de venta por unidad
VALORIZATION_RATE = 0.30  # 30% de los excedentes se valorizan

def calculate_impact(actual, predicted):
    """Calcular impacto operativo y económico"""
    excess = np.maximum(0, predicted - actual)
    deficit = np.maximum(0, actual - predicted)
    
    waste_cost = np.sum(excess) * UNIT_COST
    lost_sales = np.sum(deficit) * (SELLING_PRICE - UNIT_COST)
    
    # Baseline: producir la media histórica
    baseline_prod = np.mean(actual)
    baseline_excess = np.maximum(0, baseline_prod - actual)
    baseline_waste_cost = np.sum(baseline_excess) * UNIT_COST
    
    waste_reduction = ((baseline_waste_cost - waste_cost) / baseline_waste_cost) * 100
    
    # Valorización
    valorized_units = np.sum(excess) * VALORIZATION_RATE
    recovered_value = valorized_units * UNIT_COST * 0.5
    
    return {
        'Exceso (unidades)': np.sum(excess),
        'Déficit (unidades)': np.sum(deficit),
        'Costo_Desperdicio': waste_cost,
        'Ventas_Perdidas': lost_sales,
        'Costo_Total': waste_cost + lost_sales,
        'Baseline_Waste_Cost': baseline_waste_cost,
        'Reduccion_Desperdicio (%)': waste_reduction,
        'Ahorro_Economico': baseline_waste_cost - waste_cost,
        'Unidades_Valorizadas': valorized_units,
        'Valor_Recuperado': recovered_value
    }

In [ ]:
# Resultados de los modelos (actualizar con valores reales)
model_results = {
    'Random Forest': {'MAE': 45.2, 'MAPE': 12.3, 'Exceso': 1250, 'Ahorro': 4250, 'Reduccion': 28.5},
    'Gradient Boosting': {'MAE': 42.1, 'MAPE': 11.2, 'Exceso': 1180, 'Ahorro': 5120, 'Reduccion': 32.1},
    'XGBoost': {'MAE': 38.5, 'MAPE': 9.8, 'Exceso': 1050, 'Ahorro': 6780, 'Reduccion': 38.7},
    'Prophet': {'MAE': 48.3, 'MAPE': 13.5, 'Exceso': 1320, 'Ahorro': 3890, 'Reduccion': 24.3}
}

impact_df = pd.DataFrame(model_results).T
impact_df = impact_df.sort_values('MAE')
print("Impacto Económico por Modelo")
print(impact_df.to_string())

In [ ]:
# Gráfico: Reducción de Desperdicio vs Ahorro
fig, ax1 = plt.subplots(figsize=(10, 6))

x = np.arange(len(impact_df))
width = 0.35

bars1 = ax1.bar(x - width/2, impact_df['Reduccion'], width, label='Reducción de Desperdicio (%)', color='green')
ax1.set_ylabel('Reducción de Desperdicio (%)', color='green')
ax1.tick_params(axis='y', labelcolor='green')

ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, impact_df['Ahorro'], width, label='Ahorro Económico ($)', color='blue')
ax2.set_ylabel('Ahorro Económico ($)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

ax1.set_xticks(x)
ax1.set_xticklabels(impact_df.index, rotation=45)
ax1.set_title('Impacto por Modelo: Reducción de Desperdicio vs Ahorro')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()# Validación Cruzada Temporal con XGBoost
print("\n" + "=" * 60)
print("VALIDACIÓN CRUZADA TEMPORAL")
print("=" * 60)

from xgboost import XGBRegressor

# Preparar features
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
for lag in [1, 2, 3, 7]:
    df[f'lag_{lag}'] = df['demanda_real'].shift(lag)
df = df.dropna().reset_index(drop=True)

feature_cols = ['dia_semana', 'mes', 'lag_1', 'lag_2', 'lag_3', 'lag_7']
X = df[feature_cols].values
y = df['demanda_real'].values

# Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train_cv, X_val_cv = X[train_idx], X[val_idx]
    y_train_cv, y_val_cv = y[train_idx], y[val_idx]
    
    model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42, verbosity=0)
    model.fit(X_train_cv, y_train_cv)
    y_pred_cv = model.predict(X_val_cv)
    
    mae = mean_absolute_error(y_val_cv, y_pred_cv)
    cv_scores.append(mae)
    print(f"Fold {fold+1}: MAE = {mae:.2f}")

print(f"\n MAE promedio CV: {np.mean(cv_scores):.2f} (+/- {np.std(cv_scores):.2f})")

In [ ]:
# Gráfico de validación cruzada
plt.figure(figsize=(10, 5))
plt.plot(range(1, 6), cv_scores, marker='o', linestyle='-', color='steelblue', linewidth=2, markersize=8)
plt.axhline(y=np.mean(cv_scores), color='red', linestyle='--', label=f'Media: {np.mean(cv_scores):.2f}')
plt.fill_between(range(1, 6), np.mean(cv_scores) - np.std(cv_scores), 
                 np.mean(cv_scores) + np.std(cv_scores), alpha=0.2, color='red')
plt.title('Validación Cruzada Temporal - MAE por Fold')
plt.xlabel('Fold')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# RESUMEN FINAL
print("\n" + "=" * 60)
print("RESUMEN DE VALIDACIÓN Y MÉTRICAS")
print("=" * 60)

best_model = impact_df.loc[impact_df['MAE'].idxmin()]
print(f"\n MEJOR MODELO: {best_model.name}")
print(f"    MAE: {best_model['MAE']:.2f}")
print(f"    MAPE: {best_model['MAPE']:.1f}%")
print(f"    Reducción de Desperdicio: {best_model['Reduccion']:.1f}%")
print(f"    Ahorro Económico: ${best_model['Ahorro']:,.2f}")
print(f"\n Validación Cruzada Temporal (XGBoost): MAE = {np.mean(cv_scores):.2f} ± {np.std(cv_scores):.2f}")